# House Prices: EDA, Feature Engineering & Model Comparison

This notebook walks through the full modeling process for the Kaggle
**House Prices — Advanced Regression Techniques** competition, reusing the
tested pipeline code in `src/` rather than duplicating logic here.

**Before running:** download `train.csv` and `test.csv` from the
[competition data page](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)
and place them in `../data/raw/`.

Sections:
1. Load data & quick look
2. Missing values overview
3. Target variable distribution
4. Outliers
5. Correlation with SalePrice
6. Build the preprocessing + feature engineering pipeline
7. Baseline model comparison (linear vs. tree-based)
8. Hyperparameter tuning for XGBoost
9. Holdout evaluation
10. Feature importance
11. Generate the Kaggle submission
12. Bonus: a simple two-model blend


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBRegressor

from src import config, utils
from src.data_loader import load_raw_data
from src.preprocessing import remove_training_outliers
from src.pipeline import build_preprocessing_pipeline, build_xgb_pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100


## 1. Load data & quick look

In [ ]:
train_df, test_df = load_raw_data()
print(f"train: {train_df.shape}, test: {test_df.shape}")
train_df.head()


## 2. Missing values overview

Which columns have missing values, and how many?

In [ ]:
missing = (
    train_df.isna().sum().sort_values(ascending=False)
    .loc[lambda s: s > 0]
)
missing_pct = (missing / len(train_df) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 8))
sns.barplot(x=missing_pct.values, y=missing_pct.index, color="#4C72B0", ax=ax)
ax.set_xlabel("% missing")
ax.set_title("Missing values by column (training set)")
plt.tight_layout()
plt.show()

print("Most of these are NOT data quality problems: a missing PoolQC means")
print('"no pool", a missing GarageType means "no garage", etc. See')
print("src/preprocessing.py::MissingValueImputer for how each is handled.")


## 3. Target variable distribution

`SalePrice` is right-skewed, as home prices usually are — a handful of very expensive homes pull the mean well above the median. We model `log1p(SalePrice)` instead, which is far closer to normally distributed and lets a squared-error loss behave sensibly (errors on cheap and expensive homes are penalized proportionally, not absolutely).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(train_df["SalePrice"], kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title(f"SalePrice (skew = {train_df['SalePrice'].skew():.2f})")

log_price = np.log1p(train_df["SalePrice"])
sns.histplot(log_price, kde=True, ax=axes[1], color="#55A868")
axes[1].set_title(f"log1p(SalePrice) (skew = {log_price.skew():.2f})")

plt.tight_layout()
plt.show()


## 4. Outliers

Dean De Cock's original paper on this dataset flags two homes with `GrLivArea` above 4,000 sq ft that sold for **less** than $300,000 — almost certainly data entry issues (or highly unusual sales) rather than genuine market signal, and they're influential enough to visibly distort a fitted regression line if left in.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
outlier_mask = (train_df["GrLivArea"] > config.GRLIVAREA_OUTLIER_THRESHOLD) & \
               (train_df["SalePrice"] < config.GRLIVAREA_OUTLIER_SALEPRICE_THRESHOLD)

sns.scatterplot(data=train_df, x="GrLivArea", y="SalePrice",
                 hue=outlier_mask, palette={False: "#4C72B0", True: "#C44E52"}, ax=ax, legend=False)
ax.set_title("GrLivArea vs SalePrice (red = removed outliers)")
plt.tight_layout()
plt.show()

train_df = remove_training_outliers(train_df)
print(f"Training shape after outlier removal: {train_df.shape}")


## 5. Correlation with SalePrice

Top numeric features most linearly correlated with the target.

In [ ]:
numeric_cols = train_df.select_dtypes(include="number").columns
corr = train_df[numeric_cols].corr(numeric_only=True)["SalePrice"].drop("SalePrice")
top_corr = corr.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(x=corr[top_corr.index].values, y=top_corr.index, color="#4C72B0", ax=ax)
ax.set_xlabel("Correlation with SalePrice")
ax.set_title("Top 15 numeric features by |correlation| with SalePrice")
plt.tight_layout()
plt.show()


## 6. Build the preprocessing + feature engineering pipeline

All of the imputation, ordinal-encoding, feature-engineering and encoding logic lives in `src/`, tested independently in `tests/` — here we just assemble it and prepare `X`/`y`.

In [ ]:
X = train_df.drop(columns=[config.TARGET, config.ID_COL])
y = np.log1p(train_df[config.TARGET])

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.1, random_state=config.RANDOM_STATE
)
print(f"X_train: {X_train.shape}, X_holdout: {X_holdout.shape}")


## 7. Baseline model comparison

Before committing to XGBoost, it's worth checking how much a well-tuned gradient boosting model is actually buying us over simpler baselines. Linear models get the scaled preprocessing variant; tree models don't need scaling.

In [ ]:
baseline_models = {
    "Ridge": (Ridge(alpha=10.0), True),
    "Lasso": (Lasso(alpha=0.0005, max_iter=50000), True),
    "ElasticNet": (ElasticNet(alpha=0.0005, l1_ratio=0.5, max_iter=50000), True),
    "RandomForest": (RandomForestRegressor(n_estimators=300, random_state=config.RANDOM_STATE, n_jobs=-1), False),
    "GradientBoosting": (GradientBoostingRegressor(random_state=config.RANDOM_STATE), False),
    "XGBoost (default)": (XGBRegressor(random_state=config.RANDOM_STATE, n_jobs=-1), False),
}

from sklearn.pipeline import Pipeline

results = {}
for name, (model, needs_scaling) in baseline_models.items():
    pre = build_preprocessing_pipeline(scale_numeric=needs_scaling)
    pipe = Pipeline(steps=[*pre.steps, ("model", model)])
    scores = utils.cv_rmse(pipe, X_train, y_train, folds=5)
    results[name] = scores
    print(f"{name:20s}  CV RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})")

comparison_df = pd.DataFrame(results)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
comparison_df.mean().sort_values().plot(kind="barh", xerr=comparison_df.std(), ax=ax, color="#4C72B0")
ax.set_xlabel("CV RMSE (log SalePrice, lower is better)")
ax.set_title("Baseline model comparison (5-fold CV)")
plt.tight_layout()
plt.show()


## 8. Hyperparameter tuning for XGBoost

This mirrors `src/train.py` — a `RandomizedSearchCV` over the grid defined in `src/config.py`. This is the same code path the training script uses; the notebook version uses a smaller `n_iter` so it's quick to explore interactively. For the real run, use `python -m src.train` from the command line.

In [ ]:
pipeline = build_xgb_pipeline()

search = RandomizedSearchCV(
    pipeline,
    param_distributions=config.XGB_PARAM_GRID,
    n_iter=15,
    cv=5,
    scoring=utils.rmse_scorer,
    n_jobs=-1,
    random_state=config.RANDOM_STATE,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print(f"Best CV RMSE (log SalePrice): {-search.best_score_:.5f}")


## 9. Holdout evaluation

A final, honest check on the 10% of training data the search never touched.

In [ ]:
best_pipeline = search.best_estimator_
holdout_pred_log = best_pipeline.predict(X_holdout)

holdout_rmse_log = utils.rmse(y_holdout, holdout_pred_log)
holdout_rmse_dollars = utils.rmse(np.expm1(y_holdout), np.expm1(holdout_pred_log))

print(f"Holdout RMSE (log SalePrice): {holdout_rmse_log:.5f}")
print(f"Holdout RMSE (SalePrice, $):  {holdout_rmse_dollars:,.0f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(np.expm1(y_holdout), np.expm1(holdout_pred_log), alpha=0.6, color="#4C72B0")
lims = [np.expm1(y_holdout).min(), np.expm1(y_holdout).max()]
ax.plot(lims, lims, "--", color="gray")
ax.set_xlabel("Actual SalePrice")
ax.set_ylabel("Predicted SalePrice")
ax.set_title("Holdout: predicted vs. actual")
plt.tight_layout()
plt.show()


## 10. Feature importance

Refit on all available training data, then inspect what the model leaned on most.

In [ ]:
xgb_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
final_pipeline = build_xgb_pipeline(**xgb_params)
final_pipeline.fit(X, y)

feature_names = utils.get_output_feature_names(final_pipeline, X.sample(min(200, len(X)), random_state=config.RANDOM_STATE))
importances = final_pipeline.named_steps["model"].feature_importances_
n = min(len(feature_names), len(importances))

importance_df = (
    pd.DataFrame({"feature": feature_names[:n], "importance": importances[:n]})
    .sort_values("importance", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 8))
sns.barplot(data=importance_df, x="importance", y="feature", color="#4C72B0", ax=ax)
ax.set_title("Top 20 most important features")
plt.tight_layout()
plt.show()


## 11. Generate the Kaggle submission

Equivalent to running `python -m src.predict` from the command line (after `python -m src.train` has saved the pipeline) — done inline here for a self-contained notebook.

In [ ]:
X_test = test_df.drop(columns=[config.ID_COL])
test_predictions = np.expm1(final_pipeline.predict(X_test))

submission = pd.DataFrame({
    config.ID_COL: test_df[config.ID_COL],
    config.TARGET: test_predictions,
})
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission.to_csv(config.SUBMISSION_FILE, index=False)
submission.head()


## 12. Bonus: a simple two-model blend

Averaging a regularized linear model with a gradient boosting model often shaves a little more off the RMSE than either model alone, since the two make somewhat different kinds of errors. This is a lightweight illustration, evaluated on the same holdout split — not part of the saved production pipeline in `src/`.

In [ ]:
lasso_pre = build_preprocessing_pipeline(scale_numeric=True)
lasso_pipeline = Pipeline(steps=[*lasso_pre.steps, ("model", Lasso(alpha=0.0005, max_iter=50000))])
lasso_pipeline.fit(X_train, y_train)
lasso_holdout_pred = lasso_pipeline.predict(X_holdout)

blend_pred = 0.5 * holdout_pred_log + 0.5 * lasso_holdout_pred
blend_rmse = utils.rmse(y_holdout, blend_pred)

print(f"XGBoost alone   holdout RMSE: {holdout_rmse_log:.5f}")
print(f"Lasso alone     holdout RMSE: {utils.rmse(y_holdout, lasso_holdout_pred):.5f}")
print(f"50/50 blend     holdout RMSE: {blend_rmse:.5f}")


## Summary

- Cleaned and imputed the data without leaking test-set statistics into training.
- Encoded ordinal quality columns to preserve their ordering; one-hot encoded nominal categories.
- Engineered ~20 additional features on top of the raw columns.
- Compared linear and tree-based baselines before committing to XGBoost.
- Tuned XGBoost with cross-validated random search and checked generalization on an untouched holdout split.
- Generated a ready-to-submit `outputs/submission.csv`.

See `README.md` for the command-line equivalent (`src/train.py`, `src/predict.py`) and the test suite in `tests/`.